# Stage 4 — Driver-Gated Routing

Replaces any-alteration routing with driver-gated routing.
Reads `core_score.parquet`, `layer3_continuous.parquet`, and
`mutations_collapsed.parquet` + `fusions_gene_level.parquet`.
Writes **`outputs/flags_with_driver.parquet`**.

Key decision: alteration signal fires only when a *driver* event is present
(`any_driver | oncogene_hit | tsg_hit` from mutations;
`max_confidence == high` as fusion-driver proxy).
Abundance-tracking genes always sort on `core_score` alone — the flag never
touches them.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

OUTPUTS = Path("outputs")

In [ ]:
# Continuous alteration probabilities (Layer 3)
flags_cont = pd.read_parquet(OUTPUTS / "layer3_continuous.parquet")
flags_cont["model_id"] = flags_cont["model_id"].str.upper()

# Driver status from mutations (collapsed to gene level)
mut = pd.read_parquet("../../cleaned_track_data/mutations_collapsed.parquet")
mut["model_id"] = mut["model_id"].str.upper()

# Fusion gene-level calls
fus = pd.read_parquet("../../cleaned_track_data/fusions_gene_level.parquet")
fus["model_id"] = fus["model_id"].str.upper()

print("mut driver cols:", [c for c in mut.columns if "driver" in c or "hit" in c])
print("mut shape:", mut.shape)
print("fus shape:", fus.shape)

In [ ]:
# Confirm driver flag counts
print("any_driver True:", mut.any_driver.sum())
print("oncogene_hit True:", mut.oncogene_hit.sum())
print("tsg_hit True:", mut.tsg_hit.sum())

# Fusion driver proxy: high-confidence fusions only
print("\nfusion max_confidence:", fus.max_confidence.value_counts().to_dict())

In [ ]:
# Per-(model_id, ensg_id) mutation driver flag
mut_driver = mut[["model_id","ensg_id","any_driver","oncogene_hit","tsg_hit"]].copy()
mut_driver["mut_driver"] = (
    mut_driver["any_driver"].fillna(False)
    | mut_driver["oncogene_hit"].fillna(False)
    | mut_driver["tsg_hit"].fillna(False)
)

# Fusion driver proxy
fus_driver = fus[["model_id","ensg_id"]].copy()
fus_driver["fusion_driver"] = fus["max_confidence"] == "high"

# Merge onto continuous flags
flags = (
    flags_cont
    .merge(mut_driver[["model_id","ensg_id","mut_driver"]], on=["model_id","ensg_id"], how="left")
    .merge(fus_driver, on=["model_id","ensg_id"], how="left")
)
flags["mut_driver"]    = flags["mut_driver"].fillna(False)
flags["fusion_driver"] = flags["fusion_driver"].fillna(False)

# Any-alteration gate (p > 0.5)
flags["has_alteration"] = (flags["p_mutation"] > 0.5) | (flags["p_fusion"] > 0.5)

# Driver-gated gate
flags["has_driver_alteration"] = flags["mut_driver"] | flags["fusion_driver"]

print("any-alteration lines:", flags.has_alteration.sum())
print("driver-alteration lines:", flags.has_driver_alteration.sum())
print("driver/any ratio:", round(flags.has_driver_alteration.sum() / flags.has_alteration.sum(), 3))

In [ ]:
# Routing function (reference implementation — vectorised version used for bulk output)
def route_driver_gated(gene, core_df, flags_df, regime_df):
    """
    Returns predictions for one gene sorted by driver-gated rank.
    abundance_tracking: sort by core_score only.
    activation_driven / unknown: sort driver_flag first, then core_score.
    """
    rc = regime_df.set_index("ensg_id").loc[gene, "class"] if gene in regime_df.ensg_id.values else "unknown"
    preds = core_df[core_df.ensg_id == gene].merge(
        flags_df[flags_df.ensg_id == gene][["model_id","has_driver_alteration"]],
        on="model_id", how="left"
    )
    preds["has_driver_alteration"] = preds["has_driver_alteration"].fillna(False)
    if rc == "abundance_tracking":
        preds = preds.sort_values("core_score", ascending=False)
        preds["rank_basis"] = "core_score"
    else:
        preds = preds.sort_values(["has_driver_alteration","core_score"], ascending=[False,False])
        preds["rank_basis"] = np.where(preds.has_driver_alteration, "driver_flag+score", "score_only")
    preds["regime"] = rc
    return preds

In [ ]:
# Leak check: abundance-tracking genes must only produce rank_basis == 'core_score'
# (vectorised, uses predictions_with_confidence.parquet written in Stage 6)
pred = pd.read_parquet(OUTPUTS / "predictions_with_confidence.parquet")
abund_bases = pred[pred["class"]=="abundance_tracking"]["rank_basis"].unique()
print("rank_basis values for abundance_tracking:", abund_bases)
assert list(abund_bases) == ["core_score"], "LEAK: driver flag reached abundance_tracking genes"

In [ ]:
# Save
out_path = OUTPUTS / "flags_with_driver.parquet"
flags.to_parquet(out_path, index=False)
print("Written:", out_path)
print("Shape:", flags.shape)
print("Columns:", flags.columns.tolist())